# Netflix Shows Data ETL on GCP
This notebook is working on a ETL pipeline design for netflix shows and movies data from Kaggle, enriching the data from the open source API in OMDb.

*Source*: [Netflix TV Shows and Movies](https://www.kaggle.com/datasets/victorsoeiro/netflix-tv-shows-and-movies)

### 1. Import Necessary Libraries

In [3]:
import requests
import json
import pandas as pd

### 2. Read the CSV File and Import the Data as a Pandas Dataframe

In [4]:
df_titles = pd.read_csv('/home/jasonzelin/data-analytics-portfolio/netflix_data_etl_on_gcp/data/titles.csv')
df_credits = pd.read_csv('/home/jasonzelin/data-analytics-portfolio/netflix_data_etl_on_gcp/data/credits.csv')

In [5]:
print('Titles:\n') 
print(df_titles.info(), '\n')
print('Credits:\n') 
print(df_credits.info())

Titles:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5850 entries, 0 to 5849
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    5850 non-null   object 
 1   title                 5849 non-null   object 
 2   type                  5850 non-null   object 
 3   description           5832 non-null   object 
 4   release_year          5850 non-null   int64  
 5   age_certification     3231 non-null   object 
 6   runtime               5850 non-null   int64  
 7   genres                5850 non-null   object 
 8   production_countries  5850 non-null   object 
 9   seasons               2106 non-null   float64
 10  imdb_id               5447 non-null   object 
 11  imdb_score            5368 non-null   float64
 12  imdb_votes            5352 non-null   float64
 13  tmdb_popularity       5759 non-null   float64
 14  tmdb_score            5539 non-null   float64
dtypes: float64(5

### 3. Enriching the data from Kaggle Dataset with OMDb API Data

In [6]:
my_api_key = 'fd7f3858'
base_url = 'http://www.omdbapi.com/'
title = ''

params = {
    'apikey': my_api_key,
    't': title
}

In [11]:
titles_list = list(df_titles['title'])
omdb_data = []

for i in titles_list:
    params['t'] = i
    response = requests.get(base_url, params=params)
    data = response.json()
    if data.get("Error") == "Request limit reached!":
        print("⚠️ API daily limit reached. Breaking loop.")
        break
    omdb_data.append(data)

⚠️ API daily limit reached. Breaking loop.


In [ ]:
tb_removed = [{'Response': 'False', 'Error': 'Movie not found!'},{'Response': 'False', 'Error': 'Request limit reached!'}]
omdb_data_cleaned = [i for i in omdb_data if i not in tb_removed]

In [9]:
omdb_data_cleaned

[{'Title': 'Taxi Driver',
  'Year': '1976',
  'Rated': 'R',
  'Released': '09 Feb 1976',
  'Runtime': '114 min',
  'Genre': 'Crime, Drama',
  'Director': 'Martin Scorsese',
  'Writer': 'Paul Schrader',
  'Actors': 'Robert De Niro, Jodie Foster, Cybill Shepherd',
  'Plot': 'A mentally unstable veteran works as a nighttime taxi driver in New York City, where the perceived decadence and sleaze fuels his urge for violent action.',
  'Language': 'English, Spanish',
  'Country': 'United States',
  'Awards': 'Nominated for 4 Oscars. 22 wins & 21 nominations total',
  'Poster': 'https://m.media-amazon.com/images/M/MV5BZDNhMGYwM2UtMTdlZS00MGQ1LWI2YzAtODY5YWI1MjYyNzRmXkEyXkFqcGc@._V1_SX300.jpg',
  'Ratings': [{'Source': 'Internet Movie Database', 'Value': '8.2/10'},
   {'Source': 'Rotten Tomatoes', 'Value': '89%'},
   {'Source': 'Metacritic', 'Value': '94/100'}],
  'Metascore': '94',
  'imdbRating': '8.2',
  'imdbVotes': '989,092',
  'imdbID': 'tt0075314',
  'Type': 'movie',
  'DVD': 'N/A',
  'B

In [ ]:
omdb_data_cleaned_df = pd.read_json(json.dumps(omdb_data_cleaned))
omdb_data_cleaned_df.to_csv('/home/jasonzelin/data-analytics-portfolio/netflix_data_etl_on_gcp/data/omdb_data_batch1.csv', index=False)

/tmp/ipykernel_7172/2200419326.py:1: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  omdb_data_cleaned_csv = pd.read_json(json.dumps(omdb_data_cleaned))


In [27]:
comparison_df = df_titles.merge(right=omdb_data_cleaned_df, how='left', left_on='title', right_on='Title')[['title', 'Title']]
print(f'{comparison_df[comparison_df['Title'].isnull()].shape[0]} titles out of {comparison_df.shape[0]} from titles.csv still missing from OMDB API data.')

4958 titles out of 5860 from titles.csv still missing from OMDB API data.


In [28]:
5860-4958

902